# LeafScan: train a plant-disease CNN **from scratch**

This notebook:
1. builds a **combined dataset** from 5 sources (PlantVillage, PlantDoc, PlantWild, CCMT, Plant Pathology 2021), with ~130k images, ~90 classes, cross-dataset de-duplication and a group-aware split,
2. trains **LeafResNet** (our own CNN, random initialisation) on the GPU,
3. trains a **pretrained MobileNetV3** baseline on the same data for comparison,
4. calibrates and exports both to ONNX in the format the LeafScan API loads, with test metrics per data source.

**Runtime → Change runtime type → GPU (T4).** On Kaggle, enable *Internet* and a GPU accelerator.

Everything is saved to Google Drive (`MyDrive/leafscan`) on Colab, or `/kaggle/working/leafscan` on Kaggle. If the session disconnects, **just run all cells again**: the dataset is restored from the saved archive and training resumes from the last epoch.

| Step | Time (T4) |
|---|---|
| Build dataset (first run only) | ~45–75 min |
| LeafResNet from scratch, 50 epochs | ~3–4 h |
| MobileNetV3 baseline, 12 epochs | ~40 min |
| Export + test evaluation | ~10 min |


## 1. Setup

In [ ]:
import os, sys, json, subprocess, shutil, pathlib
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST, WORK = '/content/drive/MyDrive/leafscan', '/content'
elif IN_KAGGLE:
    PERSIST, WORK = '/kaggle/working/leafscan', '/tmp'
else:
    PERSIST, WORK = os.path.abspath('leafscan_persist'), os.path.abspath('.')
DATA = f'{WORK}/leafscan_data'
CODE = f'{WORK}/leafscan_code'
os.makedirs(PERSIST, exist_ok=True); os.makedirs(CODE, exist_ok=True)
os.chdir(WORK)
print('persist ->', PERSIST, '| data ->', DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install onnx onnxruntime onnxscript pyarrow

## 2. Code
(Generated from `ml/scratch/` in the repo. Run these cells as-is.)

In [ ]:
%%writefile leafscan_code/taxonomy.py
"""Unified label space across all datasets.

Every source label maps to one unified key in PlantVillage style ("Crop___Condition")
so existing knowledge-base entries keep working, or to None (dropped).

Merge rule: only merge labels that name the same disease (same pathogen). Labels
that are ambiguous are dropped rather than guessed. Merges that are reasonable
but not certain are listed in UNCERTAIN so they can be reviewed.
"""
from __future__ import annotations

BACKGROUND = "Background_without_leaves"

# ---------------------------------------------------------------- PlantVillage
# Folder names are already unified keys.
PLANTVILLAGE = {k: k for k in [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    BACKGROUND, "Blueberry___healthy", "Cherry___Powdery_mildew", "Cherry___healthy",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot", "Corn___Common_rust", "Corn___Northern_Leaf_Blight",
    "Corn___healthy", "Grape___Black_rot", "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Raspberry___healthy", "Soybean___healthy",
    "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus", "Tomato___healthy",
]}

# ---------------------------------------------------------------- PlantDoc (folder names)
PLANTDOC = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry___healthy",
    "Corn Gray leaf spot": "Corn___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn___Northern_Leaf_Blight",
    "Corn rust leaf": "Corn___Common_rust",
    "Peach leaf": "Peach___healthy",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___healthy",
    "Soyabean leaf": "Soybean___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato leaf": "Tomato___healthy",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape leaf black rot": "Grape___Black_rot",
    "grape leaf": "Grape___healthy",
}

# ---------------------------------------------------------------- PlantWild v1 (classes.txt names)
PLANTWILD = {
    "apple black rot": "Apple___Black_rot",
    "apple leaf": "Apple___healthy",
    "apple mosaic virus": "Apple___Mosaic_virus",
    "apple rust": "Apple___Cedar_apple_rust",
    "apple scab": "Apple___Apple_scab",
    "banana leaf": "Banana___healthy",
    "banana panama disease": "Banana___Panama_disease",
    "basil downy mildew": "Basil___Downy_mildew",
    "basil leaf": "Basil___healthy",
    "bean halo blight": "Bean___Halo_blight",
    "bean leaf": "Bean___healthy",
    "bean mosaic virus": "Bean___Mosaic_virus",
    "bean rust": "Bean___Rust",
    "bell pepper leaf": "Pepper,_bell___healthy",
    "bell pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "blueberry leaf": "Blueberry___healthy",
    "blueberry rust": "Blueberry___Rust",
    "broccoli downy mildew": "Broccoli___Downy_mildew",
    "broccoli leaf": "Broccoli___healthy",
    "cabbage alternaria leaf spot": "Cabbage___Alternaria_leaf_spot",
    "cabbage leaf": "Cabbage___healthy",
    "carrot cavity spot": None,           # a root disease, not a leaf disease
    "cauliflower alternaria leaf spot": "Cauliflower___Alternaria_leaf_spot",
    "cauliflower leaf": "Cauliflower___healthy",
    "celery anthracnose": "Celery___Anthracnose",
    "celery early blight": "Celery___Early_blight",
    "celery leaf": "Celery___healthy",
    "cherry leaf": "Cherry___healthy",
    "cherry leaf spot": "Cherry___Leaf_spot",
    "cherry powdery mildew": "Cherry___Powdery_mildew",
    "citrus canker": "Citrus___Canker",
    "citrus greening disease": "Orange___Haunglongbing_(Citrus_greening)",
    "coffee leaf": "Coffee___healthy",
    "coffee leaf rust": "Coffee___Leaf_rust",
    "corn gray leaf spot": "Corn___Cercospora_leaf_spot Gray_leaf_spot",
    "corn leaf": "Corn___healthy",
    "corn northern leaf blight": "Corn___Northern_Leaf_Blight",
    "corn rust": "Corn___Common_rust",
    "corn smut": None,                    # galls on ears/stalks, not a leaf diagnosis
    "cucumber angular leaf spot": "Cucumber___Angular_leaf_spot",
    "cucumber bacterial wilt": "Cucumber___Bacterial_wilt",
    "cucumber leaf": "Cucumber___healthy",
    "cucumber powdery mildew": "Cucumber___Powdery_mildew",
    "eggplant cercospora leaf spot": "Eggplant___Cercospora_leaf_spot",
    "eggplant leaf": "Eggplant___healthy",
    "garlic leaf": "Garlic___healthy",
    "garlic leaf blight": "Garlic___Leaf_blight",
    "garlic rust": "Garlic___Rust",
    "ginger leaf": "Ginger___healthy",
    "ginger leaf spot": "Ginger___Leaf_spot",
    "ginger sheath blight": "Ginger___Sheath_blight",
    "grape black rot": "Grape___Black_rot",
    "grape downy mildew": "Grape___Downy_mildew",
    "grape leaf": "Grape___healthy",
    "grape leaf spot": None,              # generic; could overlap Isariopsis leaf spot or black rot
    "grapevine leafroll disease": "Grape___Leafroll_disease",
    "lettuce downy mildew": "Lettuce___Downy_mildew",
    "lettuce leaf": "Lettuce___healthy",
    "lettuce mosaic virus": "Lettuce___Mosaic_virus",
    "maple leaf": "Maple___healthy",
    "maple tar spot": "Maple___Tar_spot",
    "peach leaf": "Peach___healthy",
    "peach leaf curl": "Peach___Leaf_curl",
    "plum leaf": "Plum___healthy",
    "plum pocket disease": "Plum___Pocket_disease",
    "potato early blight": "Potato___Early_blight",
    "potato late blight": "Potato___Late_blight",
    "potato leaf": "Potato___healthy",
    "raspberry leaf": "Raspberry___healthy",
    "rice blast": "Rice___Blast",
    "rice leaf": "Rice___healthy",
    "rice sheath blight": "Rice___Sheath_blight",
    "soybean leaf": "Soybean___healthy",
    "squash leaf": "Squash___healthy",
    "squash powdery mildew": "Squash___Powdery_mildew",
    "strawberry anthracnose": "Strawberry___Anthracnose",
    "strawberry leaf": "Strawberry___healthy",
    "strawberry leaf scorch": "Strawberry___Leaf_scorch",
    "tobacco leaf": "Tobacco___healthy",
    "tobacco mosaic virus": "Tobacco___Mosaic_virus",
    "tomato bacterial leaf spot": "Tomato___Bacterial_spot",
    "tomato early blight": "Tomato___Early_blight",
    "tomato late blight": "Tomato___Late_blight",
    "tomato leaf": "Tomato___healthy",
    "tomato leaf mold": "Tomato___Leaf_Mold",
    "tomato mosaic virus": "Tomato___Tomato_mosaic_virus",
    "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "tomato yellow leaf curl virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "zucchini yellow mosaic virus": "Squash___Zucchini_yellow_mosaic_virus",
}

# ---------------------------------------------------------------- CCMT raw data ("Crop/class" folders)
CCMT = {
    "Cashew/anthracnose": "Cashew___Anthracnose",
    "Cashew/gumosis": "Cashew___Gummosis",
    "Cashew/gummosis": "Cashew___Gummosis",
    "Cashew/healthy": "Cashew___healthy",
    "Cashew/leaf miner": "Cashew___Leaf_miner",
    "Cashew/red rust": "Cashew___Red_rust",
    "Cassava/bacterial blight": "Cassava___Bacterial_blight",
    "Cassava/brown spot": "Cassava___Brown_spot",
    "Cassava/green mite": "Cassava___Green_mite",
    "Cassava/healthy": "Cassava___healthy",
    "Cassava/mosaic": "Cassava___Mosaic_disease",
    "Maize/fall armyworm": "Corn___Fall_armyworm",
    "Maize/grasshoper": "Corn___Grasshopper_damage",
    "Maize/grasshopper": "Corn___Grasshopper_damage",
    "Maize/healthy": "Corn___healthy",
    "Maize/leaf beetle": "Corn___Leaf_beetle_damage",
    "Maize/leaf blight": "Corn___Northern_Leaf_Blight",
    "Maize/leaf spot": None,              # unspecified leaf spot; could be several pathogens
    "Maize/streak virus": "Corn___Maize_streak_virus",
    "Tomato/healthy": "Tomato___healthy",
    "Tomato/leaf blight": None,           # could be early OR late blight
    "Tomato/leaf curl": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato/septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato/verticulium wilt": "Tomato___Verticillium_wilt",
    "Tomato/verticillium wilt": "Tomato___Verticillium_wilt",
}

# ---------------------------------------------------------------- Plant Pathology 2021 (single-label only)
PLANT_PATHOLOGY = {
    "healthy": "Apple___healthy",
    "scab": "Apple___Apple_scab",
    "rust": "Apple___Cedar_apple_rust",
    # Frogeye leaf spot is the leaf phase of black rot (Botryosphaeria obtusa),
    # which is what PlantVillage's "Apple Black rot" leaf images show.
    "frog_eye_leaf_spot": "Apple___Black_rot",
    "powdery_mildew": "Apple___Powdery_mildew",
    "complex": None,                      # "several diseases", not one label
}

SOURCES = {
    "plantvillage": PLANTVILLAGE,
    "plantdoc": PLANTDOC,
    "plantwild": PLANTWILD,
    "ccmt": CCMT,
    "plantpathology": PLANT_PATHOLOGY,
}

# Lab vs real-world, used for per-domain reporting.
FIELD_SOURCES = {"plantdoc", "plantwild", "ccmt", "plantpathology"}

UNCERTAIN = {
    ("ccmt", "Maize/leaf blight"): "Assumed northern corn leaf blight (most common maize leaf blight).",
    ("ccmt", "Tomato/leaf curl"): "Assumed TYLCV (the dominant tomato leaf-curl virus in West Africa).",
    ("plantwild", "apple rust"): "Web 'apple rust' is mostly cedar-apple rust but may include other Gymnosporangium rusts.",
    ("plantdoc", "Bell_pepper leaf spot"): "PlantDoc's pepper leaf spot is treated as bacterial spot, as in the PlantDoc paper.",
}


def map_label(source: str, label: str) -> str | None:
    table = SOURCES[source]
    if label not in table:
        raise KeyError(f"unmapped {source} label: {label!r}")
    return table[label]


In [ ]:
%%writefile leafscan_code/sources.py
"""Downloaders for each dataset. Each source yields Items; bytes are fetched lazily
(in worker threads) so network downloads and image processing overlap.
"""
from __future__ import annotations

import io
import json
import os
import subprocess
import threading
import urllib.parse
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Iterator

import requests

UA = {"User-Agent": "Mozilla/5.0"}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

PLANTVILLAGE_URL = ("https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/"
                    "d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded")
PLANTDOC_TREE = "https://api.github.com/repos/pratikkayal/PlantDoc-Dataset/git/trees/master?recursive=1"
PLANTDOC_RAW = "https://raw.githubusercontent.com/pratikkayal/PlantDoc-Dataset/master/"
PLANTWILD_URL = "https://huggingface.co/datasets/uqtwei2/PlantWild/resolve/main/plantwild.zip"
CCMT_API = "https://data.mendeley.com/public-api/datasets/bwh3zbpkpv"
PP2021_TREE = "https://huggingface.co/api/datasets/timm/plant-pathology-2021/tree/main/data"
PP2021_FILE = "https://huggingface.co/datasets/timm/plant-pathology-2021/resolve/main/"


@dataclass
class Item:
    source: str
    label: str
    uid: str
    fetch: Callable[[], bytes]
    meta: dict = field(default_factory=dict)


def _needs_curl(url: str) -> bool:
    # Mendeley rejects Python HTTP clients (TLS fingerprinting) but accepts curl.
    return "mendeley.com" in url


def fetch(url: str, retries: int = 4) -> bytes:
    last = None
    for _ in range(retries):
        try:
            if _needs_curl(url):
                r = subprocess.run(["curl", "-sSfL", "--max-time", "120", "-A", UA["User-Agent"], url],
                                   capture_output=True)
                if r.returncode == 0:
                    return r.stdout
                last = RuntimeError(f"curl exit {r.returncode} for {url}: {r.stderr[:200]!r}")
            else:
                r = requests.get(url, headers=UA, timeout=60)
                if r.status_code == 200:
                    return r.content
                last = RuntimeError(f"HTTP {r.status_code} for {url}")
        except (requests.RequestException, OSError) as e:
            last = e
    raise last


def fetch_json(url: str):
    return json.loads(fetch(url))


def download(url: str, dest: Path, chunk: int = 1 << 22) -> Path:
    """Download a large file once (skipped if already complete)."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    if dest.exists():
        return dest
    print(f"    downloading {dest.name} ...", flush=True)
    if _needs_curl(url):
        subprocess.run(["curl", "-sSfL", "--retry", "5", "-C", "-", "-A", UA["User-Agent"], "-o", str(tmp), url],
                       check=True)
    else:
        with requests.get(url, headers=UA, stream=True, timeout=120) as r:
            r.raise_for_status()
            size, done = int(r.headers.get("Content-Length", 0)), 0
            with open(tmp, "wb") as fh:
                for block in r.iter_content(chunk):
                    fh.write(block)
                    done += len(block)
                    if size and done % (chunk * 64) < chunk:
                        print(f"    {dest.name}: {done / 1e9:.2f}/{size / 1e9:.2f} GB", flush=True)
    tmp.replace(dest)
    return dest


class RangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file via HTTP range requests (for smoke tests)."""

    def __init__(self, url: str):
        r = requests.head(url, headers=UA, allow_redirects=True, timeout=60)
        self.url, self.size, self.pos = r.url, int(r.headers["Content-Length"]), 0

    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def readinto(self, b):
        if self.pos >= self.size:
            return 0
        end = min(self.size, self.pos + len(b)) - 1
        data = requests.get(self.url, headers={**UA, "Range": f"bytes={self.pos}-{end}"}, timeout=120).content
        b[:len(data)] = data
        self.pos += len(data)
        return len(data)


def _zip_items(source: str, zip_src, label_of: Callable[[str], str | None]) -> Iterator[Item]:
    """Items from a (local or remote) zip; each worker thread gets its own handle."""
    local = threading.local()

    def handle():
        if not hasattr(local, "zf"):
            src = zip_src if isinstance(zip_src, (str, Path)) else zip_src()
            local.zf = zipfile.ZipFile(src if isinstance(src, (str, Path)) else io.BufferedReader(src, 1 << 20))
        return local.zf

    for info in handle().infolist():
        n = info.filename
        if info.is_dir() or "__MACOSX" in n or not n.lower().endswith(IMG_EXT):
            continue
        label = label_of(n)
        if label is None:
            continue
        yield Item(source, label, n, (lambda name=n: handle().read(name)))


# ---------------------------------------------------------------- sources
def plantvillage(raw: Path, smoke: bool, local_zip: str | None = None) -> Iterator[Item]:
    src = local_zip or download(PLANTVILLAGE_URL, raw / "plantvillage.zip")
    yield from _zip_items("plantvillage", src, lambda n: n.split("/")[-2])


def plantdoc(raw: Path, smoke: bool) -> Iterator[Item]:
    tree = fetch_json(PLANTDOC_TREE)["tree"]
    for t in tree:
        p = t["path"]
        parts = p.split("/")
        if t["type"] != "blob" or len(parts) != 3 or parts[0] not in ("train", "test"):
            continue
        if not p.lower().endswith(IMG_EXT):
            continue
        url = PLANTDOC_RAW + urllib.parse.quote(p)
        yield Item("plantdoc", parts[1], p, (lambda u=url: fetch(u)), {"official_split": parts[0]})


def plantwild(raw: Path, smoke: bool) -> Iterator[Item]:
    src = (lambda: RangeFile(PLANTWILD_URL)) if smoke else download(PLANTWILD_URL, raw / "plantwild.zip")

    def label_of(n: str):
        parts = n.split("/")
        return parts[-2] if len(parts) >= 3 and parts[-3] == "images" else None

    yield from _zip_items("plantwild", src, label_of)


def ccmt(raw: Path, smoke: bool) -> Iterator[Item]:
    """CCMT *raw* images only (the augmented release has transformed copies of the same
    photos in train and test). Mendeley's API lists at most 1000 files per folder."""
    folders = fetch_json(f"{CCMT_API}/folders/1")
    by_id = {f["id"]: f for f in folders}

    def path(f):
        out = []
        while f:
            out.append(f["name"])
            f = by_id.get(f.get("parent_id"))
        return "/".join(reversed(out))

    for f in sorted(folders, key=path):
        p = path(f)
        if not p.startswith("Raw Data/CCMT Dataset/") or p.count("/") != 3:
            continue
        label = "/".join(p.split("/")[2:])  # e.g. "Cassava/mosaic"
        files = fetch_json(f"{CCMT_API}/files?folder_id={f['id']}&version=1")
        for x in files:
            if not x["filename"].lower().endswith(IMG_EXT):
                continue
            url = x["content_details"]["download_url"]
            yield Item("ccmt", label, f"{label}/{x['filename']}", (lambda u=url: fetch(u)))


def plantpathology(raw: Path, smoke: bool) -> Iterator[Item]:
    """Plant Pathology 2021 (FGVC8) via the timm parquet mirror; single-label images only.
    Shards are downloaded one at a time and deleted after use (16 GB total)."""
    import pyarrow.parquet as pq

    shards = sorted(x["path"] for x in fetch_json(PP2021_TREE) if x["path"].endswith(".parquet"))
    if smoke:
        shards = shards[-1:]
    for shard in shards:
        url = PP2021_FILE + shard
        if smoke:
            pf = pq.ParquetFile(io.BufferedReader(RangeFile(url), 1 << 22))
            groups = [0]
        else:
            local = download(url, raw / "pp2021" / Path(shard).name)
            pf = pq.ParquetFile(local)
            groups = range(pf.num_row_groups)
        for g in groups:
            table = pf.read_row_group(g, columns=["image", "label_names", "image_id"]).to_pylist()
            for row in table:
                names = row["label_names"]
                if len(names) != 1:
                    continue  # multi-label images are ambiguous for single-label training
                data = row["image"]["bytes"]
                yield Item("plantpathology", names[0], row["image_id"], (lambda d=data: d))
        if not smoke:
            os.remove(local)


REGISTRY = {
    "plantvillage": plantvillage,
    "plantdoc": plantdoc,
    "plantwild": plantwild,
    "ccmt": ccmt,
    "plantpathology": plantpathology,
}


In [ ]:
%%writefile leafscan_code/build_dataset.py
"""Build the combined multi-source dataset.

    python build_dataset.py --out /content/leafscan_data              # full (Colab/Kaggle)
    python build_dataset.py --out ./smoke --smoke --max-per-class 12  # tiny local test

Steps
  1. Download each source, map labels (taxonomy.py), resize to short side 256,
     re-encode as RGB JPEG, compute MD5 + perceptual hashes (8 orientations).
  2. Drop exact duplicates; group near-duplicates (pHash distance <= 1 under any
     flip/rotation) across ALL sources; drop groups whose labels disagree.
  3. Drop classes with fewer than --min-images images.
  4. Group-aware stratified split (train/val/test = 80/10/10 per class x source),
     so copies of an image can never sit on both sides of a split.
Outputs: images/, manifest.csv, classes.json, stats.json
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import random
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

import sources
from taxonomy import FIELD_SOURCES, SOURCES, UNCERTAIN, map_label

SHORT_SIDE = 256
MAX_LONG = 512


# ---------------------------------------------------------------- hashing
def _dct(n):
    k = np.arange(n)[:, None]
    m = np.cos(np.pi * (2 * np.arange(n)[None, :] + 1) * k / (2 * n)) * np.sqrt(2 / n)
    m[0] /= np.sqrt(2)
    return m


_D = _dct(32)


def phashes(img: Image.Image) -> list[int]:
    """64-bit DCT perceptual hash for each of the 8 flips/rotations."""
    g = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    out = []
    for k in range(4):
        for flip in (False, True):
            v = np.rot90(g, k)
            v = v[:, ::-1] if flip else v
            c = (_D @ v @ _D.T)[:8, :8].ravel()
            bits = np.concatenate([[False], c[1:] > np.median(c[1:])])
            out.append(int("".join("1" if b else "0" for b in bits), 2))
    return out


# ---------------------------------------------------------------- per-image processing
def process(item: sources.Item, out_dir: Path) -> dict | None:
    try:
        data = item.fetch()
        img = Image.open(io.BytesIO(data))
        if img.format == "JPEG":
            img.draft("RGB", (SHORT_SIDE * 2, SHORT_SIDE * 2))  # fast DCT-domain downscale
        img = ImageOps.exif_transpose(img).convert("RGB")
        w, h = img.size
        if min(w, h) < 64:
            return None
        s = SHORT_SIDE / min(w, h)
        nw, nh = round(w * s), round(h * s)
        if max(nw, nh) > MAX_LONG:  # extreme aspect ratios: centre-crop the long side
            img = img.resize((nw, nh), Image.BILINEAR)
            cw, ch = min(nw, MAX_LONG), min(nh, MAX_LONG)
            l, t = (nw - cw) // 2, (nh - ch) // 2
            img = img.crop((l, t, l + cw, t + ch))
        else:
            img = img.resize((nw, nh), Image.BILINEAR if s < 1 else Image.BICUBIC)
        uid = hashlib.sha1(f"{item.source}/{item.uid}".encode()).hexdigest()[:16]
        rel = f"images/{item.source}/{uid}.jpg"
        img.save(out_dir / rel, quality=90)
        return {"path": rel, "source": item.source, "source_label": item.label,
                "key": map_label(item.source, item.label), "md5": hashlib.md5(data).hexdigest(),
                "phash": phashes(img), "orig_w": w, "orig_h": h, **item.meta}
    except Exception as e:  # corrupt image / network failure: skip, but count it
        return {"error": f"{type(e).__name__}: {e}", "source": item.source, "uid": item.uid}


def collect(source: str, out: Path, args) -> list[dict]:
    done_file = out / f"records_{source}.jsonl"
    if done_file.exists() and (out / f".done_{source}").exists():
        print(f"[{source}] already built, reusing")
        return [json.loads(l) for l in done_file.open(encoding="utf-8")]
    (out / "images" / source).mkdir(parents=True, exist_ok=True)
    kw = {"local_zip": args.plantvillage_zip} if source == "plantvillage" else {}
    items = sources.REGISTRY[source](out / "raw", args.smoke, **kw)

    per_label: Counter = Counter()
    records, errors, t0 = [], [], time.time()
    with ThreadPoolExecutor(args.workers) as pool, done_file.open("w", encoding="utf-8") as fh:
        futures = []
        for it in items:
            if it.label not in SOURCES[source]:
                raise KeyError(f"[{source}] label {it.label!r} missing from taxonomy.py")
            if SOURCES[source][it.label] is None:
                continue
            if args.max_per_class and per_label[it.label] >= args.max_per_class:
                continue
            per_label[it.label] += 1
            futures.append(pool.submit(process, it, out))
            if len(futures) >= args.workers * 8:  # bound memory: drain in order
                for f in futures[: args.workers * 4]:
                    _take(f.result(), records, errors, fh)
                futures = futures[args.workers * 4:]
                if len(records) % 2000 < args.workers * 4:
                    print(f"  [{source}] {len(records)} images ({len(records) / (time.time() - t0):.0f}/s)",
                          flush=True)
        for f in futures:
            _take(f.result(), records, errors, fh)
    (out / f".done_{source}").touch()
    print(f"[{source}] {len(records)} images, {len(errors)} errors, {(time.time() - t0) / 60:.1f} min",
          flush=True)
    if errors:
        (out / f"errors_{source}.json").write_text(json.dumps(errors[:200], indent=2))
    return records


def _take(r, records, errors, fh):
    if r is None:
        return
    if "error" in r:
        errors.append(r)
    else:
        records.append(r)
        fh.write(json.dumps(r) + "\n")


# ---------------------------------------------------------------- dedup + grouping
class DSU:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a


def group_near_duplicates(recs: list[dict], max_dist: int = 1) -> list[int]:
    """Union images whose pHash (any orientation) is within max_dist of another's canonical hash.
    Pigeonhole: with distance <= 1, at least one 32-bit half matches exactly."""
    n = len(recs)
    dsu = DSU(n)
    canon = np.array([r["phash"][0] for r in recs], dtype=np.uint64)
    buckets = [defaultdict(list), defaultdict(list)]
    for i, h in enumerate(canon):
        buckets[0][int(h) >> 32].append(i)
        buckets[1][int(h) & 0xFFFFFFFF].append(i)
    for i, r in enumerate(recs):
        for v in r["phash"]:
            for half, key in ((0, v >> 32), (1, v & 0xFFFFFFFF)):
                cand = buckets[half].get(key, ())
                if len(cand) > 500:  # degenerate hash (blank image); skip to avoid giant groups
                    continue
                for j in cand:
                    if j != i and bin(v ^ int(canon[j])).count("1") <= max_dist:
                        dsu.union(i, j)
    return [dsu.find(i) for i in range(n)]


# ---------------------------------------------------------------- split
def split_groups(recs, groups, seed=42, fracs=(0.8, 0.1, 0.1)):
    """Assign whole groups to splits, stratified by (key, majority source)."""
    rng = random.Random(seed)
    members = defaultdict(list)
    for i, g in enumerate(groups):
        members[g].append(i)
    strata = defaultdict(list)
    for g, idx in members.items():
        src = Counter(recs[i]["source"] for i in idx).most_common(1)[0][0]
        strata[(recs[idx[0]]["key"], src)].append(g)
    assign = {}
    for _, gs in sorted(strata.items()):
        rng.shuffle(gs)
        total = sum(len(members[g]) for g in gs)
        n_test, n_val = round(total * fracs[2]), round(total * fracs[1])
        if total >= 10:  # make sure tiny strata still get >= 1 val/test image
            n_test, n_val = max(n_test, 1), max(n_val, 1)
        c_test = c_val = 0
        for g in gs:
            if c_test < n_test:
                assign[g], c_test = "test", c_test + len(members[g])
            elif c_val < n_val:
                assign[g], c_val = "val", c_val + len(members[g])
            else:
                assign[g] = "train"
    return [assign[g] for g in groups]


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--sources", default=",".join(sources.REGISTRY))
    ap.add_argument("--workers", type=int, default=16)
    ap.add_argument("--min-images", type=int, default=100, help="drop classes smaller than this")
    ap.add_argument("--max-per-class", type=int, default=0, help="cap per source label (smoke tests)")
    ap.add_argument("--smoke", action="store_true", help="stream from remote zips, no big downloads")
    ap.add_argument("--plantvillage-zip", default=None, help="use a local PlantVillage zip")
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    recs: list[dict] = []
    for s in args.sources.split(","):
        recs += collect(s, out, args)
    n_raw = len(recs)

    # 1. exact duplicates (same file bytes): keep the first occurrence
    seen, kept, exact_dups = set(), [], 0
    for r in recs:
        if r["md5"] in seen:
            exact_dups += 1
            (out / r["path"]).unlink(missing_ok=True)
        else:
            seen.add(r["md5"])
            kept.append(r)
    recs = kept

    # 2. near-duplicate groups; drop groups with conflicting labels
    groups = group_near_duplicates(recs)
    keys_in = defaultdict(set)
    for r, g in zip(recs, groups):
        keys_in[g].add(r["key"])
    conflict = {g for g, ks in keys_in.items() if len(ks) > 1}
    conflict_examples = [[r["source"] + ":" + r["source_label"] for r, g2 in zip(recs, groups) if g2 == g]
                         for g in list(conflict)[:15]]
    recs_groups = [(r, g) for r, g in zip(recs, groups) if g not in conflict]
    n_conflict = len(recs) - len(recs_groups)
    group_sizes = Counter(g for _, g in recs_groups)
    group_sources = defaultdict(set)
    for r, g in recs_groups:
        group_sources[g].add(r["source"])
    cross_source = sum(1 for s in group_sources.values() if len(s) > 1)

    # 3. minimum class size
    counts = Counter(r["key"] for r, _ in recs_groups)
    small = {k: n for k, n in counts.items() if n < args.min_images}
    recs_groups = [(r, g) for r, g in recs_groups if r["key"] not in small]
    recs = [r for r, _ in recs_groups]
    groups = [g for _, g in recs_groups]

    # 4. split
    splits = split_groups(recs, groups)
    classes = sorted({r["key"] for r in recs})
    index = {k: i for i, k in enumerate(classes)}
    with (out / "manifest.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["path", "label", "key", "source", "source_label", "split", "group", "domain"])
        for r, g, s in zip(recs, groups, splits):
            w.writerow([r["path"], index[r["key"]], r["key"], r["source"], r["source_label"], s, g,
                        "field" if r["source"] in FIELD_SOURCES else "lab"])

    per_class = defaultdict(lambda: defaultdict(Counter))
    for r, s in zip(recs, splits):
        per_class[r["key"]][r["source"]][s] += 1
    stats = {
        "images_downloaded": n_raw,
        "exact_duplicates_removed": exact_dups,
        "near_duplicate_groups_multi": sum(1 for n in group_sizes.values() if n > 1),
        "cross_source_duplicate_groups": cross_source,
        "label_conflict_images_dropped": n_conflict,
        "label_conflict_examples": conflict_examples,
        "classes_dropped_too_small": small,
        "final_images": len(recs),
        "final_classes": len(classes),
        "split_sizes": Counter(splits),
        "per_source": Counter(r["source"] for r in recs),
        "uncertain_mappings_used": {f"{s}:{l}": note for (s, l), note in UNCERTAIN.items()
                                    if any(r["source"] == s and r["source_label"] == l for r in recs)},
        "per_class": {k: {s: dict(c) for s, c in v.items()} for k, v in sorted(per_class.items())},
    }
    (out / "classes.json").write_text(json.dumps(classes, indent=2))
    (out / "stats.json").write_text(json.dumps(stats, indent=2, default=dict))
    print(json.dumps({k: v for k, v in stats.items() if k not in ("per_class", "label_conflict_examples")},
                     indent=2, default=dict))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile leafscan_code/model.py
"""LeafResNet: a CNN designed for this project and trained from random initialisation.

ResNet-style backbone with the "bag of tricks" refinements that matter most when
training from scratch:
  * deep 3-conv stem (ResNet-C) instead of one 7x7 conv
  * average-pool shortcut on downsampling (ResNet-D), so no information is skipped
  * squeeze-and-excitation channel attention in every block
  * stochastic depth (drop-path) for regularisation
  * zero-initialised last BN in each residual branch (blocks start as identity)
Head: global average pool -> dropout -> linear. A GAP+linear head makes class
activation maps exact, so the API's heatmaps work unchanged.

Both models return (logits, final_feature_map).
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

CONFIGS = {
    # widths per stage, blocks per stage
    "s": ((64, 128, 256, 512), (2, 2, 3, 2)),
    "m": ((64, 128, 256, 512), (2, 3, 4, 2)),
}


def conv_bn(cin, cout, k=3, stride=1, act=True):
    layers = [nn.Conv2d(cin, cout, k, stride, k // 2, bias=False), nn.BatchNorm2d(cout)]
    if act:
        layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)


class SqueezeExcite(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        h = max(8, c // r)
        self.fc1, self.fc2 = nn.Conv2d(c, h, 1), nn.Conv2d(h, c, 1)

    def forward(self, x):
        s = x.mean((2, 3), keepdim=True)
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(s), inplace=True)))


def drop_path(x, p: float, training: bool):
    if p == 0.0 or not training:
        return x
    keep = torch.rand(x.shape[0], 1, 1, 1, device=x.device, dtype=x.dtype) >= p
    return x * keep / (1 - p)


class Block(nn.Module):
    def __init__(self, cin, cout, stride, drop):
        super().__init__()
        self.branch = nn.Sequential(
            conv_bn(cin, cout, 3, stride),
            conv_bn(cout, cout, 3, 1, act=False),
            SqueezeExcite(cout),
        )
        nn.init.zeros_(self.branch[1][1].weight)  # zero-init gamma: block starts as identity
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(
                nn.AvgPool2d(stride, stride, ceil_mode=True) if stride > 1 else nn.Identity(),
                conv_bn(cin, cout, 1, 1, act=False),
            )
        else:
            self.short = nn.Identity()
        self.drop = drop

    def forward(self, x):
        return F.relu(self.short(x) + drop_path(self.branch(x), self.drop, self.training), inplace=True)


class LeafResNet(nn.Module):
    def __init__(self, num_classes: int, size: str = "s", drop_path_rate: float = 0.1, dropout: float = 0.3):
        super().__init__()
        widths, depths = CONFIGS[size]
        self.stem = nn.Sequential(
            conv_bn(3, 32, 3, 2), conv_bn(32, 32), conv_bn(32, widths[0]),
            nn.MaxPool2d(3, 2, 1),
        )
        total, k = sum(depths), 0
        stages, cin = [], widths[0]
        for i, (w, d) in enumerate(zip(widths, depths)):
            blocks = []
            for j in range(d):
                blocks.append(Block(cin, w, 2 if (j == 0 and i > 0) else 1, drop_path_rate * k / max(1, total - 1)))
                cin, k = w, k + 1
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        self.feat_channels = cin
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        nn.init.normal_(self.fc.weight, std=0.01)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        fmap = self.stages(self.stem(x))
        return self.fc(self.dropout(fmap.mean((2, 3)))), fmap


class PretrainedMobileNet(nn.Module):
    """Transfer-learning baseline (ImageNet MobileNetV3-Large, fully fine-tuned) for comparison."""

    def __init__(self, num_classes: int, pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large
        base = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None)
        self.features = base.features
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(960, num_classes)
        self.feat_channels = 960

    def forward(self, x):
        fmap = self.features(x)
        return self.fc(self.dropout(fmap.mean((2, 3)))), fmap


def build_model(arch: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    if arch.startswith("leafresnet"):
        size = arch.split("_")[1] if "_" in arch else "s"
        return LeafResNet(num_classes, size)
    if arch == "mobilenet_v3_pretrained":
        return PretrainedMobileNet(num_classes, pretrained)
    raise ValueError(arch)


class ExportWrapper(nn.Module):
    """Inference graph matching the API contract: RGB [0,1] in -> (calibrated logits, feature map)."""

    def __init__(self, net: nn.Module, temperature: float):
        super().__init__()
        self.net = net.eval()
        self.register_buffer("mean", torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(STD).view(1, 3, 1, 1))
        self.register_buffer("temperature", torch.tensor(float(temperature)))

    def forward(self, x):
        logits, fmap = self.net((x - self.mean) / self.std)
        return logits / self.temperature, fmap


if __name__ == "__main__":
    for arch in ("leafresnet_s", "leafresnet_m"):
        m = build_model(arch, 95)
        n = sum(p.numel() for p in m.parameters())
        logits, f = m(torch.randn(2, 3, 224, 224))
        print(arch, f"{n / 1e6:.1f}M params", tuple(logits.shape), tuple(f.shape))


In [ ]:
%%writefile leafscan_code/train.py
"""Train on the combined dataset (GPU; falls back to CPU for smoke tests).

    python train.py --data /content/leafscan_data --out /content/drive/MyDrive/leafscan/runs/scratch \
                    --arch leafresnet_s --epochs 60
    python train.py ... --arch mobilenet_v3_pretrained --epochs 15 --lr 1e-3   # transfer baseline

Pipeline: DataLoader workers only read JPEG bytes; decoding (nvJPEG) and all
augmentation run batched on the GPU, so Colab's 2 CPU cores are not the bottleneck.
Recipe: AdamW + warmup/cosine, label smoothing, mixup/cutmix, random-resized-crop,
flips/rot90, colour jitter, random erasing, class-balanced sampling (with extra
weight on real-world field photos), EMA weights, AMP, resumable checkpoints.
"""
from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.io import decode_jpeg, read_file
from torchvision.transforms import RandomResizedCrop

from model import MEAN, STD, build_model

# ---------------------------------------------------------------- data
class BytesDataset(Dataset):
    def __init__(self, root: Path, rows):
        self.root, self.rows = root, rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return read_file(str(self.root / r["path"])), int(r["label"]), i


def collate(batch):
    data, labels, idx = zip(*batch)
    return list(data), torch.tensor(labels), torch.tensor(idx)


def load_split(root: Path):
    with (root / "manifest.csv").open(encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    classes = json.loads((root / "classes.json").read_text())
    return {s: [r for r in rows if r["split"] == s] for s in ("train", "val", "test")}, classes


def decode(batch: list[torch.Tensor], device) -> list[torch.Tensor]:
    if device.type == "cuda":
        return decode_jpeg(batch, device=device)
    return [decode_jpeg(b) for b in batch]


class GpuAugment:
    """Batched augmentation on uint8 CHW images of varying size -> normalised (B,3,S,S) float."""

    def __init__(self, size: int, device, train: bool):
        self.size, self.device, self.train = size, device, train
        self.mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(STD, device=device).view(1, 3, 1, 1)
        self.rrc = RandomResizedCrop(size, scale=(0.25, 1.0), ratio=(3 / 4, 4 / 3))

    def __call__(self, imgs: list[torch.Tensor]) -> torch.Tensor:
        out = []
        for im in imgs:
            if self.train:
                i, j, h, w = self.rrc.get_params(im, self.rrc.scale, self.rrc.ratio)
                im = im[:, i:i + h, j:j + w]
                k = int(torch.randint(0, 4, (1,)))
                if k:
                    im = torch.rot90(im, k, (1, 2))
            # eval: plain resize to SxS (same as the API's preprocessing)
            out.append(F.interpolate(im[None].float(), size=(self.size, self.size), mode="bilinear",
                                     antialias=True, align_corners=False)[0])
        x = torch.stack(out) / 255.0
        if self.train:
            b = x.shape[0]
            flip = torch.rand(b, device=x.device) < 0.5
            x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
            # colour jitter: brightness, contrast, saturation
            def rnd(lo, hi):
                return torch.empty(b, 1, 1, 1, device=x.device).uniform_(lo, hi)
            x = x * rnd(0.7, 1.3)
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = (x - gray.mean((2, 3), keepdim=True)) * rnd(0.7, 1.3) + gray.mean((2, 3), keepdim=True)
            gray = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
            x = gray + (x - gray) * rnd(0.6, 1.4)
            x = x.clamp(0, 1)
        x = (x - self.mean) / self.std
        if self.train:  # random erasing (p=0.25), filled with noise
            b = x.shape[0]
            for n in torch.nonzero(torch.rand(b) < 0.25).flatten().tolist():
                eh, ew = (int(self.size * float(torch.empty(1).uniform_(0.1, 0.4))) for _ in range(2))
                ey, ex = int(torch.randint(0, self.size - eh + 1, (1,))), int(torch.randint(0, self.size - ew + 1, (1,)))
                x[n, :, ey:ey + eh, ex:ex + ew] = torch.randn(3, eh, ew, device=x.device)
        return x.contiguous(memory_format=torch.channels_last)


def mix(x, y_soft, mixup_alpha, cutmix_alpha, prob):
    """Mixup or CutMix on a batch with soft targets."""
    if np.random.rand() > prob:
        return x, y_soft
    perm = torch.randperm(x.shape[0], device=x.device)
    if np.random.rand() < 0.5 and cutmix_alpha > 0:
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        H, W = x.shape[2:]
        ch, cw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y0, y1, x0, x1 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
        x = x.clone()
        x[:, :, y0:y1, x0:x1] = x[perm, :, y0:y1, x0:x1]
        lam = 1 - (y1 - y0) * (x1 - x0) / (H * W)
    else:
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        x = lam * x + (1 - lam) * x[perm]
    return x, lam * y_soft + (1 - lam) * y_soft[perm]


# ---------------------------------------------------------------- EMA
class EMA:
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))  # warm up the average early in training
        for e, m in zip(self.model.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(m.detach(), alpha=1 - d)
            else:
                e.copy_(m)


# ---------------------------------------------------------------- eval
@torch.no_grad()
def evaluate(model, loader, aug, device, rows, n_classes, amp):
    model.eval()
    logits_all, y_all, idx_all = [], [], []
    for data, y, idx in loader:
        x = aug(decode(data, device))
        with torch.autocast(device.type, enabled=amp):
            logits, _ = model(x)
        logits_all.append(logits.float().cpu()); y_all.append(y); idx_all.append(idx)
    logits = torch.cat(logits_all)
    y = torch.cat(y_all)
    idx = torch.cat(idx_all)
    pred = logits.argmax(1)
    res = {"loss": float(F.cross_entropy(logits, y)), "acc": float((pred == y).float().mean()),
           "macro_f1": macro_f1(y.numpy(), pred.numpy(), n_classes)}
    for dom in ("lab", "field"):
        m = torch.tensor([rows[i]["domain"] == dom for i in idx.tolist()])
        if m.any():
            res[f"acc_{dom}"] = float((pred[m] == y[m]).float().mean())
    return res, logits, y


def macro_f1(y, p, n):
    f1s = []
    for c in range(n):
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        if tp + fp + fn:
            f1s.append(2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s)) if f1s else 0.0


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--arch", default="leafresnet_s")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=3, help="warmup epochs")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--smoothing", type=float, default=0.1)
    ap.add_argument("--mixup", type=float, default=0.2)
    ap.add_argument("--cutmix", type=float, default=1.0)
    ap.add_argument("--mix-prob", type=float, default=0.5)
    ap.add_argument("--ema", type=float, default=0.9995)
    ap.add_argument("--field-boost", type=float, default=1.5, help="sampling weight multiplier for field photos")
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 2))
    ap.add_argument("--max-steps-per-epoch", type=int, default=0, help="smoke tests")
    ap.add_argument("--no-pretrained", action="store_true")
    args = ap.parse_args()

    torch.manual_seed(0); np.random.seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    torch.backends.cudnn.benchmark = True
    root, out = Path(args.data), Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    splits, classes = load_split(root)
    n_classes = len(classes)
    print(f"device {device} | classes {n_classes} | " +
          " ".join(f"{k} {len(v)}" for k, v in splits.items()), flush=True)

    # class-balanced sampling (1/sqrt(n)), with real-world photos boosted
    train_rows = splits["train"]
    counts = Counter(int(r["label"]) for r in train_rows)
    w = [(1 / math.sqrt(counts[int(r["label"])])) * (args.field_boost if r["domain"] == "field" else 1.0)
         for r in train_rows]
    sampler = WeightedRandomSampler(w, num_samples=len(train_rows), replacement=True)
    kw = dict(batch_size=args.batch, num_workers=args.workers, collate_fn=collate,
              pin_memory=False, persistent_workers=args.workers > 0)
    train_dl = DataLoader(BytesDataset(root, train_rows), sampler=sampler, drop_last=True, **kw)
    val_dl = DataLoader(BytesDataset(root, splits["val"]), shuffle=False, **kw)
    aug_train, aug_eval = GpuAugment(args.img, device, True), GpuAugment(args.img, device, False)

    model = build_model(args.arch, n_classes, pretrained=not args.no_pretrained).to(device)
    model = model.to(memory_format=torch.channels_last)
    ema = EMA(model, args.ema)
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.wd},
                             {"params": no_decay, "weight_decay": 0.0}], lr=args.lr, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(enabled=amp)
    steps_per_epoch = args.max_steps_per_epoch or len(train_dl)
    total, warm = args.epochs * steps_per_epoch, int(args.warmup * steps_per_epoch)

    def lr_at(step):
        if step < warm:
            return args.lr * (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 1e-6 + 0.5 * (args.lr - 1e-6) * (1 + math.cos(math.pi * t))

    start_epoch, step, history, best = 1, 0, [], -1.0
    ckpt = out / "last.pt"
    if ckpt.exists():
        s = torch.load(ckpt, map_location=device, weights_only=False)
        model.load_state_dict(s["model"]); ema.model.load_state_dict(s["ema"])
        opt.load_state_dict(s["opt"]); scaler.load_state_dict(s["scaler"])
        start_epoch, step, history, best = s["epoch"] + 1, s["step"], s["history"], s["best"]
        print(f"resumed from epoch {s['epoch']} (best val macro-F1 {best:.4f})", flush=True)

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        t0, seen, correct, loss_sum = time.time(), 0, 0.0, 0.0
        for it, (data, y, _) in enumerate(train_dl):
            if args.max_steps_per_epoch and it >= args.max_steps_per_epoch:
                break
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            y = y.to(device, non_blocking=True)
            x = aug_train(decode(data, device))
            y_soft = F.one_hot(y, n_classes).float() * (1 - args.smoothing) + args.smoothing / n_classes
            x, y_mix = mix(x, y_soft, args.mixup, args.cutmix, args.mix_prob)
            with torch.autocast(device.type, enabled=amp):
                logits, _ = model(x)
                loss = torch.sum(-y_mix * F.log_softmax(logits.float(), 1), 1).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            ema.update(model, step)
            step += 1
            seen += len(y); loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y_mix.argmax(1)).sum().item()
            if it % 100 == 0:
                rate = seen / (time.time() - t0)
                print(f"  ep {epoch} it {it}/{steps_per_epoch} loss {loss.item():.3f} "
                      f"lr {lr_at(step):.2e} {rate:.0f} img/s", flush=True)

        val, _, _ = evaluate(ema.model, val_dl, aug_eval, device, splits["val"], n_classes, amp)
        rec = {"epoch": epoch, "train_loss": loss_sum / max(seen, 1), "train_acc": correct / max(seen, 1),
               "val_loss": val["loss"], "val_acc": val["acc"], "val_macro_f1": val["macro_f1"],
               "val_acc_lab": val.get("acc_lab"), "val_acc_field": val.get("acc_field"),
               "seconds": round(time.time() - t0, 1)}
        history.append(rec)
        print(f"[epoch {epoch}/{args.epochs}] train loss {rec['train_loss']:.3f} | val acc {val['acc']:.4f} "
              f"macroF1 {val['macro_f1']:.4f} lab {val.get('acc_lab', 0):.4f} field {val.get('acc_field', 0):.4f} "
              f"| {rec['seconds']:.0f}s", flush=True)

        meta = {"arch": args.arch, "classes": classes, "img": args.img, "args": vars(args)}
        if val["macro_f1"] > best:
            best = val["macro_f1"]
            torch.save({"ema": ema.model.state_dict(), "epoch": epoch, **meta}, out / "best.pt")
        torch.save({"model": model.state_dict(), "ema": ema.model.state_dict(), "opt": opt.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch, "step": step, "history": history,
                    "best": best, **meta}, out / "last.pt.tmp")
        os.replace(out / "last.pt.tmp", ckpt)  # atomic, so a disconnect never corrupts the checkpoint
        (out / "history.json").write_text(json.dumps(history, indent=2))

    print(f"done. best val macro-F1 {best:.4f} -> {out / 'best.pt'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile leafscan_code/export.py
"""Calibrate, export to ONNX (API format) and evaluate on the held-out test set.

    python export.py --data /content/leafscan_data --run <run dir> --out <run dir>/api_model

Writes everything the LeafScan API needs into --out:
  leafnet.onnx, cam_head.npz, labels.json, metrics.json, train_history.json, dataset_stats.json
Test metrics are computed with ONNX Runtime and the API's own PIL preprocessing,
overall, per data source and per domain (lab vs field).
"""
from __future__ import annotations

import argparse
import csv
import json
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader

from model import ExportWrapper, build_model
from train import BytesDataset, GpuAugment, collate, evaluate, load_split


def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    out, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(out)


def prf(y, p, n):
    rows = []
    for c in range(n):
        tp = int(np.sum((p == c) & (y == c))); fp = int(np.sum((p == c) & (y != c)))
        fn = int(np.sum((p != c) & (y == c)))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        rows.append((pr, rc, f1, tp + fn))
    return rows


def summary(y, probs, n):
    p = probs.argmax(1)
    top = np.argsort(-probs, 1)
    per = prf(y, p, n)
    present = [r for r in per if r[3] > 0]
    return {
        "n_images": int(len(y)),
        "accuracy": float((p == y).mean()),
        "top3_accuracy": float(np.mean([y[i] in top[i, :3] for i in range(len(y))])),
        "top5_accuracy": float(np.mean([y[i] in top[i, :5] for i in range(len(y))])),
        "macro_f1": float(np.mean([r[2] for r in present])) if present else 0.0,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True)
    ap.add_argument("--out", default=None)
    ap.add_argument("--batch", type=int, default=128)
    args = ap.parse_args()
    root, run = Path(args.data), Path(args.run)
    out = Path(args.out) if args.out else run / "api_model"
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ck = torch.load(run / "best.pt", map_location="cpu", weights_only=False)
    classes, img = ck["classes"], ck["img"]
    n = len(classes)
    net = build_model(ck["arch"], n, pretrained=False)
    net.load_state_dict(ck["ema"])
    net = net.to(device).eval()
    splits, _ = load_split(root)

    # ---- temperature scaling on validation logits
    val_dl = DataLoader(BytesDataset(root, splits["val"]), batch_size=args.batch, shuffle=False,
                        num_workers=2, collate_fn=collate)
    _, val_logits, val_y = evaluate(net, val_dl, GpuAugment(img, device, False), device, splits["val"], n,
                                    device.type == "cuda")
    T = fit_temperature(val_logits, val_y)
    ece_before = ece(F.softmax(val_logits, 1).numpy(), val_y.numpy())
    ece_after = ece(F.softmax(val_logits / T, 1).numpy(), val_y.numpy())
    print(f"temperature {T:.3f}  val ECE {ece_before:.4f} -> {ece_after:.4f}", flush=True)

    # ---- ONNX export (API contract)
    wrapper = ExportWrapper(net.cpu().float(), T).eval()
    onnx_path = out / "leafnet.onnx"
    torch.onnx.export(wrapper, torch.rand(1, 3, img, img), str(onnx_path), input_names=["image"],
                      output_names=["logits", "features"],
                      dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}, "features": {0: "batch"}},
                      opset_version=17, dynamo=False)
    np.savez(out / "cam_head.npz", weight=net.fc.weight.detach().numpy().astype(np.float32),
             bias=net.fc.bias.detach().numpy().astype(np.float32))
    (out / "labels.json").write_text(json.dumps(classes, indent=2))

    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    x = torch.rand(2, 3, img, img)
    with torch.no_grad():
        ref = wrapper(x)[0].numpy()
    diff = float(np.abs(ref - sess.run(["logits"], {"image": x.numpy()})[0]).max())
    print(f"exported {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB), torch vs onnx max diff {diff:.2e}")

    # ---- test evaluation with the API's preprocessing (PIL bilinear resize, [0,1])
    test = splits["test"]
    probs = np.zeros((len(test), n), dtype=np.float32)
    y = np.array([int(r["label"]) for r in test])
    t0 = time.time()
    for s in range(0, len(test), 64):
        batch = np.stack([
            np.asarray(Image.open(root / r["path"]).convert("RGB").resize((img, img), Image.BILINEAR),
                       dtype=np.float32).transpose(2, 0, 1) / 255.0 for r in test[s:s + 64]])
        z = sess.run(["logits"], {"image": batch})[0]
        z = np.exp(z - z.max(1, keepdims=True))
        probs[s:s + 64] = z / z.sum(1, keepdims=True)
        if s % 2048 == 0:
            print(f"  test {s}/{len(test)}", flush=True)
    secs = time.time() - t0
    lat = []
    for _ in range(20):
        t = time.perf_counter(); sess.run(["logits"], {"image": batch[:1]}); lat.append((time.perf_counter() - t) * 1000)

    pred = probs.argmax(1)
    per = prf(y, pred, n)
    cm = np.zeros((n, n), dtype=int)
    np.add.at(cm, (y, pred), 1)
    metrics = {
        "split": "test", **summary(y, probs, n),
        "weighted_f1": float(sum(r[2] * r[3] for r in per) / max(1, sum(r[3] for r in per))),
        "ece": ece(probs, y),
        "latency_ms_p50": float(np.percentile(lat, 50)), "latency_ms_p95": float(np.percentile(lat, 95)),
        "throughput_img_s": len(test) / secs, "model_size_mb": onnx_path.stat().st_size / 1e6,
        "classes": classes,
        "per_class": [{"class": c, "precision": per[i][0], "recall": per[i][1], "f1": per[i][2],
                       "support": per[i][3]} for i, c in enumerate(classes)],
        "confusion_matrix": cm.tolist(),
        "per_source": {}, "per_domain": {},
    }
    for key, field in (("per_source", "source"), ("per_domain", "domain")):
        groups = defaultdict(list)
        for i, r in enumerate(test):
            groups[r[field]].append(i)
        for g, idx in sorted(groups.items()):
            metrics[key][g] = summary(y[idx], probs[idx], n)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))

    history = json.loads((run / "history.json").read_text()) if (run / "history.json").exists() else []
    params = sum(p.numel() for p in net.parameters())
    (out / "train_history.json").write_text(json.dumps({
        "history": history, "best_val_macro_f1": max((h["val_macro_f1"] for h in history), default=None),
        "temperature": T, "val_ece_before": ece_before, "val_ece_after": ece_after,
        "config": {**ck["args"], "backbone": ("LeafResNet (trained from scratch)" if ck["arch"].startswith("leafresnet")
                                              else "MobileNetV3-Large (ImageNet pretrained, fine-tuned)"),
                   "params_millions": round(params / 1e6, 2), "label_smoothing": ck["args"]["smoothing"]},
    }, indent=2))
    stats = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})
    for s, rows in splits.items():
        for r in rows:
            stats[r["key"]][s] += 1
    (out / "dataset_stats.json").write_text(json.dumps(stats, indent=2))

    print(f"TEST accuracy {metrics['accuracy']:.4f} | macro-F1 {metrics['macro_f1']:.4f} | "
          f"top-5 {metrics['top5_accuracy']:.4f} | ECE {metrics['ece']:.4f}")
    for g, m in metrics["per_domain"].items():
        print(f"  {g:6s} acc {m['accuracy']:.4f}  macro-F1 {m['macro_f1']:.4f}  (n={m['n_images']})")
    for g, m in metrics["per_source"].items():
        print(f"  {g:15s} acc {m['accuracy']:.4f}  macro-F1 {m['macro_f1']:.4f}  (n={m['n_images']})")


if __name__ == "__main__":
    main()


## 3. Build (or restore) the combined dataset
First run downloads ~20 GB in total (Plant Pathology parquet shards are streamed and deleted one at a time),
resizes everything to a short side of 256 px and writes a ~3–4 GB archive to Drive. Later runs restore that archive in ~2 minutes.

In [ ]:
ARCHIVE = f'{PERSIST}/leafscan_data.tar'
if os.path.exists(f'{DATA}/manifest.csv'):
    print('dataset already present')
elif os.path.exists(ARCHIVE):
    print('restoring dataset from', ARCHIVE)
    !tar -xf "$ARCHIVE" -C "$WORK"
else:
    !cd leafscan_code && python -u build_dataset.py --out "$DATA" --workers 32
    print('archiving to', ARCHIVE)
    !tar --exclude='raw' --exclude='records_*' -cf "$ARCHIVE" -C "$WORK" leafscan_data
stats = json.load(open(f'{DATA}/stats.json'))
print(json.dumps({k: v for k, v in stats.items() if k not in ('per_class', 'label_conflict_examples')}, indent=1))

## 4. Train LeafResNet from scratch
Re-run this cell after a disconnect; it resumes from `last.pt`.

In [ ]:
RUN_SCRATCH = f'{PERSIST}/runs/leafresnet_s'
!cd leafscan_code && python -u train.py --data "$DATA" --out "$RUN_SCRATCH" --arch leafresnet_s --epochs 50 --batch 128 --lr 2e-3 --workers 4

## 5. Transfer-learning baseline (pretrained MobileNetV3, fine-tuned)

In [ ]:
RUN_PRE = f'{PERSIST}/runs/mobilenet_v3_pretrained'
!cd leafscan_code && python -u train.py --data "$DATA" --out "$RUN_PRE" --arch mobilenet_v3_pretrained --epochs 12 --batch 128 --lr 1e-3 --wd 0.02 --warmup 1 --workers 4

## 6. Calibrate, export to ONNX and evaluate on the test set

In [ ]:
for run in (RUN_SCRATCH, RUN_PRE):
    if os.path.exists(f'{run}/best.pt'):
        !cd leafscan_code && python -u export.py --data "$DATA" --run "$run" 

In [ ]:
import pandas as pd
rows = []
for name, run in (('LeafResNet (from scratch)', RUN_SCRATCH), ('MobileNetV3 (pretrained)', RUN_PRE)):
    p = f'{run}/api_model/metrics.json'
    if not os.path.exists(p):
        continue
    m = json.load(open(p))
    h = json.load(open(f'{run}/api_model/train_history.json'))
    row = {'model': name, 'params (M)': h['config']['params_millions'], 'test acc': m['accuracy'],
           'macro-F1': m['macro_f1'], 'top-5': m['top5_accuracy'], 'ECE': m['ece']}
    for d, dm in m['per_domain'].items():
        row[f'{d} acc'] = dm['accuracy']
    for s, sm in m['per_source'].items():
        row[f'{s} acc'] = sm['accuracy']
    rows.append(row)
pd.DataFrame(rows).set_index('model').T.round(4)

## 7. Download the model for the LeafScan app
Unzip into `plant-disease-ai/api/models/` (replacing the current files) and restart the API.

In [ ]:
for run in (RUN_SCRATCH, RUN_PRE):
    if os.path.exists(f'{run}/api_model/leafnet.onnx'):
        z = shutil.make_archive(f'{run}/leafscan_api_model', 'zip', f'{run}/api_model')
        print('saved', z)
if IN_COLAB and os.path.exists(f'{RUN_SCRATCH}/leafscan_api_model.zip'):
    from google.colab import files
    files.download(f'{RUN_SCRATCH}/leafscan_api_model.zip')